In [3]:
import pandas as pd
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch
import torch
import torch.nn.functional as F

data_folder = "./data/Training_Data"
n_inputs = 1000

fits_data = []

for i in tqdm(range(0,n_inputs)):
    fits_file = fits.open(f"{data_folder}/input_image_{i}.fits")
    fits_data.append(np.array(fits_file)[0].data)


fits_data = np.array(fits_data)

train_data = torch.load('./train_data_saved_3000_observations.pt')
test_data = torch.load('./test_data_saved_3000_observations.pt')

  0%|          | 0/1000 [00:00<?, ?it/s]

TimeoutError: [Errno 60] Operation timed out

In [ ]:
import torch
import torch.nn as nn
# Edge proximity weighting function
def edge_proximity_weights(y_true):
    """
    Compute edge proximity weights for each point.

    Args:
        y_true: Tensor of shape [batch_size, n_dim], with values in [0, 1].

    Returns:
        weights: Tensor of shape [batch_size], edge proximity weights.
    """
    # Compute proximity to edges for each coordinate
    weights = torch.maximum(y_true, 1 - y_true)

    # Combine proximity weights across all dimensions (sum, product, etc.)
    #weights = edge_proximity.sum(dim=1)  # Summing across dimensions

    # Optionally normalize the weights to avoid large values
    weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-6)
    weights = (10*weights) + 1  # Scale to ensure weights are >= 1
    return weights

class FeatureLoss(nn.Module):
    def __init__(self, alpha=1, beta=1, gamma=1):
        """
        Custom combined loss function for RA, Dec, and Flux predictions with flexible scaling factors.

        Parameters:
        - alpha, beta, gamma: Weights for the RA, Dec, and Flux loss components.
        - ra_scale, dec_scale, flux_scale: Scaling factors for each component to improve gradient balance.
        """
        super(FeatureLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, predictions, targets):
        predictions, targets = predictions.float(), targets.float()

        # Separate RA, Dec, and Flux predictions and targets
        ra_pred, ra_true = predictions[:, ::3], targets[:, ::3]
        dec_pred, dec_true = predictions[:, 1::3], targets[:, 1::3]
        flux_pred, flux_true = predictions[:, 2::3], targets[:, 2::3]

        # Calculate individual losses with scaling
        ra_loss = F.l1_loss(ra_pred, ra_true)
        #ra_loss = (F.l1_loss(ra_pred, ra_true,reduction='none') * edge_proximity_weights(ra_true)).mean()
        dec_loss = F.l1_loss(dec_pred, dec_true)
        #dec_loss = (F.l1_loss(dec_pred, dec_true,reduction='none') * edge_proximity_weights(dec_true)).mean()
        flux_loss = F.l1_loss(flux_pred, flux_true)

        # Combine losses
        combined_loss = self.alpha * ra_loss + self.beta * dec_loss + self.gamma * flux_loss

        return combined_loss


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# OpenAI generated
class EuclideanLoss(nn.Module):
    def __init__(self, alpha=3, beta=1, gamma=1):
        """
        Custom loss function combining Euclidean loss for RA and Dec
        with L1 loss for Flux predictions.

        Parameters:
        - alpha: Weight for the Euclidean loss (RA and Dec).
        - beta: Weight for the L1 loss of Flux.
        - gamma: Optional additional scaling for Flux loss (if needed).
        """
        super(EuclideanLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, predictions, targets):
        """
        Compute the Euclidean loss for RA and Dec, and L1 loss for Flux.

        Parameters:
        - predictions: Tensor of predicted RA, Dec, and Flux (batch_size x 3).
        - targets: Tensor of true RA, Dec, and Flux (batch_size x 3).

        Returns:
        - combined_loss: Weighted combination of Euclidean and L1 losses.
        """
        predictions, targets = predictions.float(), targets.float()

        # Separate RA, Dec, and Flux predictions and targets
        ra_pred, ra_true = predictions[:, 0], targets[:, 0]
        dec_pred, dec_true = predictions[:, 1], targets[:, 1]
        flux_pred, flux_true = predictions[:, 2], targets[:, 2]

        # Calculate Euclidean distance loss for RA and Dec
        euclidean_distance = torch.sqrt((ra_pred - ra_true) ** 2 + (dec_pred - dec_true) ** 2)
        euclidean_loss = euclidean_distance.mean()

        # Calculate L1 loss for Flux
        flux_loss = F.l1_loss(flux_pred, flux_true)

        # Combine losses with weights
        combined_loss = self.alpha * euclidean_loss + self.beta * flux_loss

        return combined_loss


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, BatchNorm, AttentionalAggregation, GCNConv

class GraphRegressor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=None, num_layers=4, dropout=0.3):
        super(GraphRegressor, self).__init__()

        self.dropout = dropout

        # Convolutional layers
        self.conv_layers = torch.nn.ModuleList()
        self.bn_layers = torch.nn.ModuleList()
        self.residual_layers = torch.nn.ModuleList()

        # Initial convolution layer
        self.conv_layers.append(SAGEConv(in_channels, hidden_channels))
        self.bn_layers.append(BatchNorm(hidden_channels))
        if in_channels != hidden_channels:
            # Linear layer to project input to hidden_channels for residual connection
            self.residual_layers.append(torch.nn.Linear(in_channels, hidden_channels))
        else:
            self.residual_layers.append(torch.nn.Identity())

        # Additional layers
        for _ in range(num_layers - 1):
            self.conv_layers.append(SAGEConv(hidden_channels, hidden_channels))
            self.bn_layers.append(BatchNorm(hidden_channels))
            self.residual_layers.append(torch.nn.Identity())

        self.final_conv = SAGEConv(hidden_channels, 3)

        self.att_pool = AttentionalAggregation(gate_nn=torch.nn.Linear(3, 1))


    def forward(self, x, edge_index, batch, edge_attr=None):

        for conv, bn,res in zip(self.conv_layers, self.bn_layers, self.residual_layers):
            if isinstance(res, torch.nn.Identity):
                x_res = x
            else:
                x_res = res(x)

            if edge_attr is not None:
                x = conv(x, edge_index, edge_attr)
            else:
                x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res  # Add residual connection


        out = self.final_conv(x, edge_index)
        out = F.relu(out)

        out = self.att_pool(out, batch)


        return out


In [ ]:
ensemble_size = 5

models = []
optimizers = []
criterions = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = 6
hidden_channels = 32
out_channels = 3

for _ in range(ensemble_size):
    model = GraphRegressor(in_channels, hidden_channels,out_channels,num_layers=2).to(device)
    loss_fn = EuclideanLoss().to(device)
    optimizer  = torch.optim.Adam(model.parameters())

    models.append(model)
    optimizers.append(optimizer)
    criterions.append(loss_fn)

print(device)


In [ ]:
epochs = 2000

for ep in range(epochs):
    model_losses = []
    for i,(model,optimizer,criterion) in enumerate(zip(models,optimizers,criterions)):
        model.train()
        optimizer.zero_grad()

        out = model(train_data.x, train_data.edge_index, train_data.batch)
        total_loss = criterion(out, train_data.y)
        total_loss.backward()
        optimizer.step()
        if ep % 10 == 0:
            with torch.no_grad():
                test_out = model(test_data.x, test_data.edge_index, test_data.batch)
                #val_loss = loss_fn(test_out,test_data.y)
                #scheduler.step(val_loss)
                ra_loss =  F.l1_loss(test_out[:,::3], test_data.y[:, ::3]) #* loss_fn.ra_scale
                dec_loss =  F.l1_loss(test_out[:,1::3], test_data.y[:, 1::3])# * loss_fn.dec_scale
                flux_loss = F.l1_loss(test_out[:,2::3], test_data.y[:, 2::3]) #* loss_fn.flux_scale


                model_loss = {"ra":ra_loss.item(),"dec":dec_loss.item(),"flux":flux_loss.item()}

            model_losses.append(model_loss)
    if ep % 10 == 0:
        mean_ra_loss = np.mean([x["ra"] for x in model_losses])
        mean_dec_loss = np.mean([x["dec"] for x  in model_losses])
        mean_flux_loss = np.mean([x["flux"] for x in model_losses])
        total_loss = mean_ra_loss + mean_dec_loss + mean_flux_loss
        nr_of_models = len(model_losses)
        print(f"Iteration {ep} - Mean Total Loss: {total_loss:.4f}, Mean RA Loss: {mean_ra_loss:.4f}, Mean Dec Loss: {dec_loss:.4f}, Mean Flux Loss: {flux_loss:.4f}, Number Of Models: {nr_of_models}")


